# Experiment: Training von Grund auf & Framework-Debugging

**Autor:** Nikolai Steffan
**Kontext:** Studienarbeit Kapitel 6.2.2
**Status:** ⚠️ Diagnose-Modus (Identifikation von Stagnation & Data-Split-Fehlern)

---

## 1. Einleitung und Zielsetzung

Nach dem Scheitern des Fine-Tuning-Ansatzes (siehe vorheriges Notebook) dokumentiert dieses Notebook den Versuch, ein Transformer-Modell (`BERTSmallTransformer`) von Grund auf ("from scratch") mit dem **FlowTransformer-Framework** zu trainieren.

**Ziel:** Nutzung des Frameworks wie vorgesehen, um die Ursache für die schlechte Performance zu isolieren.

**Beobachtete Probleme (gemäß Kap. 6.2.2):**
1.  **Modellstagnation:** Das Modell lernt nicht und sagt konstant die Mehrheitsklasse vorher.
2.  **Fehlerhafte Evaluation:** Die `LastRows`-Split-Methode erzeugt ein Test-Set ohne Angriffe.
3.  **Klassen-Imbalance:** Extremes Ungleichgewicht (ca. 17:1) im Rohdatensatz.

## 2. Setup und Initialisierung

Import der notwendigen Bibliotheken und der spezifischen Transformer-Implementierungen (`BERTSmallTransformer`).

In [1]:
import os
import pandas as pd
from implementations.classification_heads import LastTokenClassificationHead
from implementations.transformers.basic_transformers import BasicTransformer
from implementations.transformers.named_transformers import BERTSmallTransformer

demonstration_folder = "demonstration"

if not os.path.exists(demonstration_folder):
    os.mkdir(demonstration_folder)

2025-09-25 14:32:59.213426: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Datenbasis: UNSW-NB15

Wir verwenden den **UNSW-NB15** Datensatz als Grundlage. Im folgenden Schritt wird der Datensatz entpackt und für die Verarbeitung durch das Framework bereitgestellt. Der Datensatz kann [hier](https://staff.itee.uq.edu.au/marius/NIDS_datasets/) heruntergeladen werden

In [2]:
import zipfile

csv_path = os.path.join(demonstration_folder, "NF-UNSW-NB15-v3.csv")



print(f"Dataset is available at {csv_path}, size = {os.path.getsize(csv_path):,}")

Dataset is available at demonstration/NF-UNSW-NB15-v3.csv, size = 577,360,958


## 4. Definition des Datenschemas

Das FlowTransformer-Framework benötigt eine strikte `DatasetSpecification`. Hier definieren wir:
* **Numerische Features:** Werden normalisiert.
* **Kategorische Features:** Werden für das Embedding vorbereitet.
* **Zielvariable:** "Attack" (binäre Klassifikation gegen "Benign").

In [3]:
from framework.dataset_specification import DatasetSpecification

flow_format = DatasetSpecification(
        include_fields=['NUM_PKTS_UP_TO_128_BYTES', 'SRC_TO_DST_SECOND_BYTES', 'OUT_PKTS', 'OUT_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'DST_TO_SRC_AVG_THROUGHPUT', 'DURATION_IN', 'L4_SRC_PORT', 'ICMP_TYPE', 'PROTOCOL', 'SERVER_TCP_FLAGS', 'IN_PKTS', 'NUM_PKTS_512_TO_1024_BYTES', 'CLIENT_TCP_FLAGS', 'TCP_WIN_MAX_IN', 'NUM_PKTS_256_TO_512_BYTES', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'LONGEST_FLOW_PKT', 'L4_DST_PORT', 'MIN_TTL', 'DST_TO_SRC_SECOND_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'DURATION_OUT', 'FLOW_DURATION_MILLISECONDS', 'TCP_FLAGS', 'MAX_TTL', 'SRC_TO_DST_AVG_THROUGHPUT', 'ICMP_IPV4_TYPE', 'MAX_IP_PKT_LEN', 'RETRANSMITTED_OUT_BYTES', 'IN_BYTES', 'RETRANSMITTED_IN_BYTES', 'TCP_WIN_MAX_OUT', 'L7_PROTO', 'RETRANSMITTED_OUT_PKTS', 'RETRANSMITTED_IN_PKTS', 'FLOW_START_MILLISECONDS','FLOW_END_MILLISECONDS','SRC_TO_DST_IAT_MIN','SRC_TO_DST_IAT_MAX','SRC_TO_DST_IAT_AVG','SRC_TO_DST_IAT_STDDEV','DST_TO_SRC_IAT_MIN','DST_TO_SRC_IAT_MAX','DST_TO_SRC_IAT_AVG','DST_TO_SRC_IAT_STDDEV'],
        categorical_fields=['CLIENT_TCP_FLAGS', 'L4_SRC_PORT', 'TCP_FLAGS', 'ICMP_IPV4_TYPE', 'ICMP_TYPE', 'PROTOCOL', 'SERVER_TCP_FLAGS', 'L4_DST_PORT', 'L7_PROTO'],
        class_column="Attack",
        benign_label="Benign"
    )

## 5. Definition der Architektur

Hier wird die Pipeline zusammengesetzt.
* **Modell:** `BERTSmallTransformer` (ca. 30 Mio. Parameter).
* **Embedding:** 64-dimensionale Vektoren.

⚠️ **Hypothese aus der Arbeit:** Dieses Modell könnte für die geringe Datenmenge und das Training von Grund auf überdimensioniert sein.

In [4]:
from framework.flow_transformer_parameters import FlowTransformerParameters
from framework.flow_transformer import FlowTransformer
from implementations.input_encodings import RecordLevelEmbed
from implementations.pre_processings import StandardPreProcessing

# We use several standard component to build our transformer
pre_processing = StandardPreProcessing(n_categorical_levels=32)
encoding = RecordLevelEmbed(64)
transformer = BERTSmallTransformer()
classification_head = LastTokenClassificationHead()

# Define the transformer
ft = FlowTransformer(pre_processing=pre_processing,
                     input_encoding=encoding,
                     sequential_model=transformer,
                     classification_head=classification_head,
                     params=FlowTransformerParameters(window_size=8, mlp_layer_sizes=[128], mlp_dropout=0.1))

## 6. Laden und Splitten (Kritischer Fehler)

Laden des Datensatzes und Aufteilung in Training/Test.

❌ **Fehlerquelle (Kapitel 6.2.2):**
Der Parameter `evaluation_dataset_sampling=EvaluationDatasetSampling.LastRows` führt dazu, dass die letzten 10% der Daten als Testset genommen werden. Da der Datensatz chronologisch oder sortiert sein kann, enthält dieses Testset **keine Angriffe** (siehe Output unten: `Positive samples in eval set: 0`). Dies macht die Evaluation der Detektionsleistung unmöglich.

In [5]:
from framework.enumerations import EvaluationDatasetSampling
from IPython.display import display

df = ft.load_dataset("UNSW-NB15",
                csv_path,
                specification=flow_format,
                evaluation_dataset_sampling=EvaluationDatasetSampling.LastRows,
                evaluation_percent=0.1,
                cache_path=demonstration_folder)

display(df.iloc[:500])

Using cache file path: demonstration/UNSW-NB15_0_QdLmZHuh8yOmlGcKBEkf7hepImY0_5EjmvToFWKee8t20u0dFpVzNu4s0.feather
Reading directly from cache demonstration/UNSW-NB15_0_QdLmZHuh8yOmlGcKBEkf7hepImY0_5EjmvToFWKee8t20u0dFpVzNu4s0.feather...


,RETRANSMITTED_OUT_BYTES,DURATION_IN,MIN_TTL,IN_BYTES,OUT_PKTS,DURATION_OUT,RETRANSMITTED_IN_BYTES,DST_TO_SRC_AVG_THROUGHPUT,SRC_TO_DST_SECOND_BYTES,NUM_PKTS_UP_TO_128_BYTES,...,L7_PROTO_23,L7_PROTO_24,L7_PROTO_25,L7_PROTO_26,L7_PROTO_27,L7_PROTO_28,L7_PROTO_29,L7_PROTO_30,L7_PROTO_31,L7_PROTO_32
0,0.000000,0.000000,0.625000,0.291233,0.118030,0.000000,0.000000,0.770031,0.536183,0.195001,...,False,False,False,False,False,False,False,False,False,False
1,0.441220,0.496802,0.625000,0.494162,0.361767,0.496811,0.462598,0.657980,0.272973,0.430769,...,False,False,False,False,False,False,False,False,False,False
2,0.791846,0.667165,0.625000,0.556469,0.653691,0.667177,0.454312,0.847971,0.644755,0.686549,...,False,False,False,False,False,False,False,False,False,False
3,0.000000,0.000000,0.625000,0.291233,0.118030,0.000000,0.000000,0.793919,0.618112,0.195001,...,False,False,False,False,False,False,False,False,False,False
4,0.000000,0.000000,0.625000,0.284449,0.118030,0.000000,0.000000,0.788370,0.606954,0.195001,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.353461,0.325269,0.625000,0.382916,0.297874,0.343962,0.319891,0.702388,0.357933,0.446948,...,False,False,False,False,False,False,False,False,False,False
496,0.273134,0.432585,0.999294,0.299086,0.148937,0.443256,0.266888,0.516722,0.070595,0.335929,...,False,False,False,False,False,False,False,False,False,False
497,0.384774,0.423462,0.625000,0.405460,0.332087,0.429285,0.336771,0.663319,0.283114,0.478736,...,False,False,False,False,False,False,False,False,False,False
498,0.395387,0.435770,0.625000,0.416763,0.350035,0.440835,0.349387,0.665534,0.287095,0.496074,...,False,False,False,False,False,False,False,False,False,False


## 7. Modell-Kompilierung

Erstellen des Keras-Modells.
* **Optimizer:** Adam.
* **Loss:** Binary Crossentropy.

In [6]:
# Build the transformer model
m = ft.build_model()
m.summary()

# Compile the model
m.compile(optimizer="adam", loss='binary_crossentropy', metrics=['binary_accuracy'], jit_compile=True)

I0000 00:00:1758810822.695546    3945 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22158 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:41:00.0, compute capability: 8.6
I0000 00:00:1758810822.697417    3945 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22356 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:61:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_RETRANSMITTE… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_DURATION_IN   │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_MIN_TTL       │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_IN_BYTES      │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_OUT_PKTS      │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_DURATION_OUT  │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_RETRANSMITTE… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_DST_TO_SRC_A… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_SRC_TO_DST_S… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_NUM_PKTS_UP_… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_NUM_PKTS_102… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_MAX_TTL       │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_RETRANSMITTE… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_RETRANSMITTE… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_TCP_WIN_MAX_… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_FLOW_DURATIO… │ (None, 8, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_MIN_IP_PKT_L… │ (None, 8, 1)      │          0 │ -               

 Total params: 29,862,081 (113.91 MB)

 Trainable params: 29,862,081 (113.91 MB)

 Non-trainable params: 0 (0.00 B)

## 8. Training und Evaluation

Start des Trainings.

**Analyse der Ergebnisse (Kapitel 6.2.2):**
* Das Modell erreicht eine `Balanced Accuracy` von exakt **50.00%**.
* **TP (True Positives) = 0**: Das Modell erkennt keine Angriffe.
* Dies bestätigt die **Modellstagnation**: Das Netz hat gelernt, einfach immer "Benign" (Gutartig) vorherzusagen, was aufgrund der `LastRows`-Problematik im Testset zu einer trügerischen "hohen" Accuracy (aber 0 F1-Score) führt.

**Konsequenz:** Wechsel zu `RandomRows` Sampling, Reduktion der Modellkomplexität (`BasicTransformer`) und Bugfixing im Framework (Dropout-Layer).

In [7]:
m.compile(optimizer="adam", loss='binary_crossentropy', metrics=['binary_accuracy'], jit_compile=True)

# Get the evaluation results
eval_results: pd.DataFrame
(train_results, eval_results, final_epoch) = ft.evaluate(m, batch_size=128, epochs=5, steps_per_epoch=64, early_stopping_patience=5)

print(eval_results)

Building eval dataset...
Splitting dataset to featurewise...
Evaluation dataset is built!
Positive samples in eval set: 0
Negative samples in eval set: 236542


2025-09-25 14:34:55.051482: I external/local_xla/xla/service/service.cc:163] XLA service 0x561961d1d020 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-25 14:34:55.051508: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3090, Compute Capability 8.6
2025-09-25 14:34:55.051511: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (1): NVIDIA GeForce RTX 3090, Compute Capability 8.6
2025-09-25 14:34:55.663184: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-09-25 14:34:58.008274: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90500
2025-09-25 14:35:00.506576: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints

Epoch = 0 / 5 (early stop in 5), step = 0, loss = 0.72952, results = [array(0.72952384, dtype=float32), array(0.484375, dtype=float32)] -- elapsed (train): 0.00s
Epoch = 0 / 5 (early stop in 5), step = 23, loss = 0.83569, results = [array(0.8356949, dtype=float32), array(0.50846356, dtype=float32)] -- elapsed (train): 2.01s
Epoch = 0 / 5 (early stop in 5), step = 46, loss = 0.77375, results = [array(0.7737455, dtype=float32), array(0.5078125, dtype=float32)] -- elapsed (train): 3.92s
Epoch = 1 / 5 (early stop in 5), step = 5, loss = 0.75085, results = [array(0.7508463, dtype=float32), array(0.5074777, dtype=float32)] -- elapsed (train): 5.84s
Epoch = 1 / 5 (early stop in 5), step = 28, loss = 0.73851, results = [array(0.73850715, dtype=float32), array(0.5031922, dtype=float32)] -- elapsed (train): 7.76s
Epoch = 1 / 5 (early stop in 5), step = 51, loss = 0.72977, results = [array(0.72977424, dtype=float32), array(0.50511855, dtype=float32)] -- elapsed (train): 9.67s
Epoch = 2 / 5 (early

2025-09-25 14:36:02.756926: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-09-25 14:36:02.757115: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-09-25 14:36:03.799055: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_7', 108 bytes spill stores, 108 bytes spill loads



7387/7392 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step

2025-09-25 14:36:56.300660: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-09-25 14:36:56.300808: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-09-25 14:36:56.941580: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_7', 16 bytes spill stores, 16 bytes spill loads

2025-09-25 14:36:57.002248: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Re

7392/7392 ━━━━━━━━━━━━━━━━━━━━ 58s 7ms/step
Epoch 4 yielded predictions: (236542,), overall balanced accuracy: 50.00%, TP = 0 / 0, TN = 236,542 / 236,542
   epoch  P       N  pred_P  pred_N  TP  FP      TN  FN  bal_acc  f1
0      4  0  236542       0  236542   0   0  236542   0      0.5   0


This model can now be used for machine learning training